# 23-13 · Перемещаем файлы во временном каталоге

Практика к разделу [«Безопасно перемещаем файлы»](../../site/chapters/glava-23/23-13-peremeshaem-fajly.html). Использует настоящий пакет `safesort`.

## Reproducible local environment

```bash
git clone https://github.com/Cartesian-School/safesort.git
cd safesort
python3.14 -m venv .venv
source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name safesort-py314 --display-name "SafeSort Python 3.14"
jupyter lab
```

Select the **SafeSort Python 3.14** kernel. The diagnostic cell below must
point into this `.venv` and the cloned `src/safesort` tree.

In [ ]:
import sys
import safesort

print(sys.executable)
print(safesort.__file__)

## Цель

Вызвать настоящую `safesort.executor.apply_plan()` во временном каталоге и убедиться, что она перемещает файлы и отказывается перезаписывать уже занятое место назначения.

## Example

In [ ]:
import tempfile
from pathlib import Path

from safesort.executor import apply_plan
from safesort.models import MoveOperation, SortPlan

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

istochnik = koren / "otchet.pdf"
istochnik.write_text("содержимое отчёта", encoding="utf-8")

naznachenie = koren / "Sorted" / "documents" / "otchet.pdf"
plan = SortPlan(root=koren, operations=(MoveOperation(source=istochnik, destination=naznachenie),))

rezultaty = apply_plan(plan)
print(rezultaty)

## Проверка результата

In [ ]:
assert rezultaty[0].completed is True
assert not istochnik.exists()
assert naznachenie.exists()
assert naznachenie.read_text(encoding="utf-8") == "содержимое отчёта"
print("Верно: файл перемещён, содержимое не повреждено, исходное место пусто.")

## Эксперимент — существующий файл в месте назначения не перезаписывается

In [ ]:
istochnik2 = koren / "zametka.txt"
istochnik2.write_text("новый текст", encoding="utf-8")

naznachenie2 = koren / "Sorted" / "documents" / "zametka.txt"
naznachenie2.parent.mkdir(parents=True, exist_ok=True)
naznachenie2.write_text("уже лежавший здесь текст", encoding="utf-8")

plan2 = SortPlan(root=koren, operations=(MoveOperation(source=istochnik2, destination=naznachenie2),))
rezultaty2 = apply_plan(plan2)

assert rezultaty2[0].completed is False
assert "already exists" in rezultaty2[0].error
assert istochnik2.exists()
assert naznachenie2.read_text(encoding="utf-8") == "уже лежавший здесь текст"
print("Верно: apply_plan отказался перезаписать существующий файл.")

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def peremestit_dva(root: Path):
    # TODO: create two files, build one SortPlan, call apply_plan().
    raise NotImplementedError


## Task

Напишите `peremestit_dva(root)`: создайте a.txt и b.txt, выполните один SortPlan и верните результаты.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    test_root = Path(tmp)
    rezultaty3 = peremestit_dva(test_root)
    assert len(rezultaty3) == 2 and all(r.completed for r in rezultaty3)
    assert (test_root / "Sorted/documents/a.txt").read_text() == "A"
    assert (test_root / "Sorted/documents/b.txt").read_text() == "B"
print("Tests passed")

## Hint

Обе `MoveOperation` поместите в один tuple `SortPlan.operations`.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def peremestit_dva(root: Path):
    source_a = root / "a.txt"
    source_b = root / "b.txt"
    source_a.write_text("A", encoding="utf-8")
    source_b.write_text("B", encoding="utf-8")
    destination = root / "Sorted" / "documents"
    plan = SortPlan(
        root=root,
        operations=(
            MoveOperation(source_a, destination / "a.txt"),
            MoveOperation(source_b, destination / "b.txt"),
        ),
    )
    return apply_plan(plan)
```

</details>